# Qwen2.5-1.5B-Instruct — Few-shot ABSA Inference
Phân loại đa nhãn cảm xúc theo khía cạnh cho bình luận game Liên Quân Mobile.

- Nhãn: `0` = không nhắc, `1` = nhắc trung tính/tích cực, `2` = nhắc tiêu cực
- Kỹ thuật: Few-shot prompting (KHÔNG fine-tune), dùng model gốc `Qwen/Qwen2.5-1.5B-Instruct`
- Input: `/kaggle/input/dataset/{train,val,test}.csv`
- Output: `/kaggle/working/dev_predictions.csv`, `/kaggle/working/test_predictions.csv`

In [1]:
!pip install -q -U transformers accelerate bitsandbytes pandas tqdm scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 90.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0

## Import & cấu hình

In [5]:
import os
import re
import json
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Danh sách khía cạnh theo đúng thứ tự cột trong csv
ASPECTS = [
    "graphics",
    "matchmaking",
    "store & microtransactions",
    "technical_issue",
    "mechanics",
    "developer_support",
    "event",
    "community",
    "hero_design",
    "difficulty",
]

TRAIN_PATH = "/kaggle/input/datasets/bebbequin/dataset/train.csv"
DEV_PATH   = "/kaggle/input/datasets/bebbequin/dataset/val.csv"
TEST_PATH  = "/kaggle/input/datasets/bebbequin/dataset/test.csv"

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## Load model & tokenizer 

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
    ).to(device)

model.eval()
print("Model loaded.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


## Load dữ liệu

In [6]:
df_train = pd.read_csv(TRAIN_PATH)
df_dev   = pd.read_csv(DEV_PATH)
df_test  = pd.read_csv(TEST_PATH)

print("Train:", df_train.shape, "Dev:", df_dev.shape, "Test:", df_test.shape)
df_train.head()

Train: (4152, 11) Dev: (519, 11) Test: (519, 11)


,review,graphics,matchmaking,store & microtransactions,technical_issue,mechanics,developer_support,event,community,hero_design,difficulty
0,cơ chế tính kda thì như tệ hại afk với phá thì...,0,2,0,0,2,0,0,2,0,0
1,có này chơi thì cứ vào là bị xếp với mấy kid t...,0,2,0,0,0,0,0,2,0,0
2,nhân phẩm tài khoản đen nạp mua gương không tr...,0,2,2,0,0,2,0,2,0,0
3,cập nhật xong chơi được 2 trận lại văng game. ...,0,0,0,2,2,0,0,0,0,0
4,set điểm mới như tệ hại không hiểu gà luôn ?????,0,0,0,0,2,2,0,0,0,0


## Few-shot examples
Dùng 10 mẫu để prompt few-shot (lấy từ train set).

In [7]:
FEWSHOT_RAW = [
    {
        "review": "cơ chế tính kda thì như tệ hại afk với phá thì đánh một tí thì không tố cáo được nói chung cơ chế ghép như tệ hại lệch rank vô cùng",
        "labels": [0,2,0,0,2,0,0,2,0,0]
    },
    {
        "review": "có này chơi thì cứ vào là bị xếp với mấy kid trẻ trâu mới vô trận đã chửi mà vô game thì đánh ngu hơn gì nữa ý như rằng thắng 1 trận là thua cả 10 trận",
        "labels": [0,2,0,0,0,0,0,2,0,0]
    },
    {
        "review": "nhân phẩm tài khoản đen nạp mua gương không trúng tôi không nói, giờ cày rank thì gặp lũ phá với feed cay không chịu được, bị hạ sao mà còn tố cáo không được",
        "labels": [0,2,2,0,0,2,0,2,0,0]
    },
    {
        "review": "cập nhật xong chơi được 2 trận lại văng game. lâu lâu lại văng game khi đang pick tướng hoặc load trận làm trừ rất nhiều uy tín.",
        "labels": [0,0,0,2,2,0,0,0,0,0]
    },
    {
        "review": "set điểm mới như tệ hại không hiểu gà luôn ?????",
        "labels": [0,0,0,0,2,2,0,0,0,0]
    },
    {
        "review": "mong game sửa lỗi vô trận chứ gần 6 tháng rồi chưa sửa nổi cái lỗi nguyên ngày mệt mỏi vô game làm trận cúp trận đầu văng tệ hại game xong vô lại thì thua luôn còn bị trừ uy tín do tố cáo quá mệt mỏi với game!",
        "labels": [0,0,0,2,2,2,0,0,0,0]
    },
    {
        "review": "mẹ game lag như gì mà chơi xong bị trừ uy tín yêu cầu game sửa lỗi và trả uy tín",
        "labels": [0,0,0,2,2,2,0,0,0,0]
    },
    {
        "review": "chơi cứ bị chuỗi thua vô lý đề nghị gà tệ hại lại lỗi ghép trận được nhiều pro hơn",
        "labels": [0,2,0,0,0,2,0,0,0,0]
    },
    {
        "review": "rất hay lỗi",
        "labels": [0,0,0,2,0,0,0,0,0,0]
    },
    {
        "review": "cơ chế như tệ hại. cộng thêm thắng 1 trận thua 2 trận",
        "labels": [0,2,0,0,2,0,0,0,0,0]
    },
]

def labels_to_json_str(labels):
    d = {ASPECTS[i]: labels[i] for i in range(len(ASPECTS))}
    return json.dumps(d, ensure_ascii=False)

FEWSHOT_EXAMPLES = [
    (ex["review"], labels_to_json_str(ex["labels"])) for ex in FEWSHOT_RAW
]
print(FEWSHOT_EXAMPLES[0])

('cơ chế tính kda thì như tệ hại afk với phá thì đánh một tí thì không tố cáo được nói chung cơ chế ghép như tệ hại lệch rank vô cùng', '{"graphics": 0, "matchmaking": 2, "store & microtransactions": 0, "technical_issue": 0, "mechanics": 2, "developer_support": 0, "event": 0, "community": 2, "hero_design": 0, "difficulty": 0}')


## Xây dựng prompt few-shot

In [8]:
ASPECT_LIST_STR = ", ".join(ASPECTS)

SYSTEM_PROMPT = f"""Bạn là một hệ thống phân loại ABSA (Aspect-Based Sentiment Analysis) cho bình luận tiếng Việt về game Liên Quân Mobile trên CH Play.
Với mỗi bình luận, hãy gán nhãn cho TẤT CẢ {len(ASPECTS)} khía cạnh sau: {ASPECT_LIST_STR}.
Với mỗi khía cạnh, chọn đúng một trong ba nhãn:
0 = không nhắc đến khía cạnh này
1 = có nhắc đến, cảm xúc trung tính hoặc tích cực
2 = có nhắc đến, cảm xúc tiêu cực

Chỉ trả lời bằng một object JSON duy nhất, đúng {len(ASPECTS)} khóa theo đúng tên khía cạnh ở trên, giá trị là số nguyên 0, 1 hoặc 2. Không giải thích, không thêm text nào khác ngoài JSON."""

def build_messages(review_text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex_review, ex_json in FEWSHOT_EXAMPLES:
        messages.append({"role": "user", "content": f"Bình luận: {ex_review}"})
        messages.append({"role": "assistant", "content": ex_json})
    messages.append({"role": "user", "content": f"Bình luận: {review_text}"})
    return messages

## Hàm inference + parsing output

In [9]:
def parse_prediction(text):
    """Trích JSON từ output của model, trả về list nhãn theo đúng thứ tự ASPECTS.
    Nếu parse lỗi hoặc thiếu khóa, mặc định = 0."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    labels = [0] * len(ASPECTS)
    if not match:
        return labels
    json_str = match.group(0)
    try:
        obj = json.loads(json_str)
    except Exception:
        return labels
    for i, aspect in enumerate(ASPECTS):
        val = obj.get(aspect, 0)
        try:
            val = int(val)
        except Exception:
            val = 0
        if val not in (0, 1, 2):
            val = 0
        labels[i] = val
    return labels

@torch.no_grad()
def predict_one(review_text, max_new_tokens=150):
    messages = build_messages(review_text)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return parse_prediction(text), text

# test nhanh trên 1 câu
sample_review = df_dev["review"].iloc[0]
labels, raw = predict_one(sample_review)
print("Review:", sample_review)
print("Raw output:", raw)
print("Parsed:", labels)

Review: mẹ con tướng mới yếu vô cùng trả lại tướng cho bố
Raw output: {"graphics": 0, "matchmaking": 0, "store & microtransactions": 0, "technical_issue": 0, "mechanics": 2, "developer_support": 0, "event": 0, "community": 0, "hero_design": 0, "difficulty": 0}
Parsed: [0, 0, 0, 0, 2, 0, 0, 0, 0, 0]


## Chạy inference trên toàn bộ tập Dev và Test

In [10]:
def run_inference(df, desc="inference"):
    all_preds = []
    for review in tqdm(df["review"].tolist(), desc=desc):
        labels, _ = predict_one(review)
        all_preds.append(labels)
    pred_df = pd.DataFrame(all_preds, columns=ASPECTS)
    result_df = pd.concat([df[["review"]].reset_index(drop=True), pred_df], axis=1)
    return result_df

dev_preds = run_inference(df_dev, desc="Dev inference")
dev_preds.to_csv(f"{OUT_DIR}/dev_predictions.csv", index=False)
dev_preds.head()

Dev inference:   0%|          | 0/519 [00:00<?, ?it/s]

,review,graphics,matchmaking,store & microtransactions,technical_issue,mechanics,developer_support,event,community,hero_design,difficulty
0,mẹ con tướng mới yếu vô cùng trả lại tướng cho bố,0,0,0,0,2,0,0,0,0,0
1,ghép trận chi team bạn toàn mấy thằng chơi hay...,0,2,0,0,2,0,0,0,0,0
2,ghép trận như tệ hại,0,2,0,0,2,0,0,0,0,0
3,tạm được,0,0,0,0,0,0,0,0,0,0
4,"giật lag, game tệ hại.",0,0,0,2,0,0,0,0,0,0


In [11]:
test_preds = run_inference(df_test, desc="Test inference")
test_preds.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
test_preds.head()

Test inference:   0%|          | 0/519 [00:00<?, ?it/s]

,review,graphics,matchmaking,store & microtransactions,technical_issue,mechanics,developer_support,event,community,hero_design,difficulty
0,game thấy mình đánh hay toàn ghép chung với tụ...,0,2,0,0,2,0,0,0,0,0
1,tranh lane được mvp ra tố cáo không phát hiện ...,0,0,0,0,2,0,0,2,0,0
2,gắn bó với game cũng đã lâu và đầu tư rất nhiề...,0,0,2,0,0,0,0,0,0,0
3,sao không cho đăng nhập bằng facebook nữa,0,0,0,2,0,0,0,0,0,0
4,tôi cập nhật xong mất tài khoản luôn đang rank...,0,0,0,2,2,0,0,0,0,0


## 9. Đánh giá cơ bản trên dev và test

In [12]:
from sklearn.metrics import f1_score, classification_report

def evaluate(df_true, df_pred, split_name="dev"):
    y_true = df_true[ASPECTS].values
    y_pred = df_pred[ASPECTS].values
    macro_f1_per_aspect = []
    print(f"===== Kết quả trên tập {split_name} =====")
    for i, aspect in enumerate(ASPECTS):
        f1 = f1_score(y_true[:, i], y_pred[:, i], average="macro", zero_division=0)
        macro_f1_per_aspect.append(f1)
        print(f"{aspect:30s} macro-F1: {f1:.4f}")
    overall = sum(macro_f1_per_aspect) / len(macro_f1_per_aspect)
    print(f"\n>>> Mean Macro-F1 ({split_name}): {overall:.4f}")
    return overall

# Chỉ chạy nếu cột nhãn có sẵn trong file gốc (val.csv / test.csv)
if all(a in df_dev.columns for a in ASPECTS):
    evaluate(df_dev, dev_preds, "dev")

if all(a in df_test.columns for a in ASPECTS):
    evaluate(df_test, test_preds, "test")

===== Kết quả trên tập dev =====
graphics                       macro-F1: 0.3715
matchmaking                    macro-F1: 0.5277
store & microtransactions      macro-F1: 0.4897
technical_issue                macro-F1: 0.5370
mechanics                      macro-F1: 0.3954
developer_support              macro-F1: 0.4330
event                          macro-F1: 0.3714
community                      macro-F1: 0.4469
hero_design                    macro-F1: 0.3251
difficulty                     macro-F1: 0.5656

>>> Mean Macro-F1 (dev): 0.4463
===== Kết quả trên tập test =====
graphics                       macro-F1: 0.4018
matchmaking                    macro-F1: 0.5373
store & microtransactions      macro-F1: 0.4292
technical_issue                macro-F1: 0.5377
mechanics                      macro-F1: 0.4449
developer_support              macro-F1: 0.3977
event                          macro-F1: 0.4040
community                      macro-F1: 0.4241
hero_design                    macro